In [2]:
# =========================================================
# PIECE 1: Setup, Library Installation, and Data Loading
# =========================================================

# Install libraries not preinstalled in Colab
!pip install catboost lightgbm optuna shap imbalanced-learn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# -----------------------------------------------------------
# Mount Google Drive (recommended: keep datasets in Drive so
# they persist across Colab sessions instead of re-uploading)
# -----------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

# Update these paths to match where you upload your files in Drive
URL_DATASET_PATH = '/content/drive/MyDrive/phishing_project/PhiUSIIL_URL_Dataset.csv'
EMAIL_DATASET_PATH = '/content/drive/MyDrive/phishing_project/phishing_email_dataset.csv'

# -----------------------------------------------------------
# Load datasets
# -----------------------------------------------------------
df_url = pd.read_csv(URL_DATASET_PATH)
df_email = pd.read_csv(EMAIL_DATASET_PATH)

print("=== URL Dataset ===")
print("Shape:", df_url.shape)
print(df_url.head())
print("\nColumns:", list(df_url.columns))

print("\n=== Email Dataset ===")
print("Shape:", df_email.shape)
print(df_email.head())
print("\nColumns:", list(df_email.columns))

# -----------------------------------------------------------
# Quick sanity checks
# -----------------------------------------------------------
print("\n=== URL Dataset Info ===")
print(df_url.info())
print("\nMissing values (URL):\n", df_url.isnull().sum()[df_url.isnull().sum() > 0])

print("\n=== Email Dataset Info ===")
print(df_email.info())
print("\nMissing values (Email):\n", df_email.isnull().sum()[df_email.isnull().sum() > 0])

ERROR: Exception:
Traceback (most recent call last):
  File "/home/rohit/Downloads/RVU_Phishing_Detection/venv/lib/python3.12/site-packages/pip/_vendor/urllib3/response.py", line 438, in _error_catcher
    yield
  File "/home/rohit/Downloads/RVU_Phishing_Detection/venv/lib/python3.12/site-packages/pip/_vendor/urllib3/response.py", line 561, in read
    data = self._fp_read(amt) if not fp_closed else b""
           ^^^^^^^^^^^^^^^^^^
  File "/home/rohit/Downloads/RVU_Phishing_Detection/venv/lib/python3.12/site-packages/pip/_vendor/urllib3/response.py", line 527, in _fp_read
    return self._fp.read(amt) if amt is not None else self._fp.read()
           ^^^^^^^^^^^^^^^^^^
  File "/home/rohit/Downloads/RVU_Phishing_Detection/venv/lib/python3.12/site-packages/pip/_vendor/cachecontrol/filewrapper.py", line 98, in read
    data: bytes = self.__fp.read(amt)
                  ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/http/client.py", line 484, in read
    s = self.fp.read(amt)
        ^

ModuleNotFoundError: No module named 'seaborn'

In [ ]:
# =========================================================
# PIECE 2: Load Correct Email Dataset + Clean Both Datasets
# =========================================================

# -----------------------------------------------------------
# 2a. Download the correct labeled email dataset via kagglehub
#     (Kaggle: subhajournal/phishingemails)
# -----------------------------------------------------------
!pip install kagglehub -q
import kagglehub

# This will prompt you to authenticate with your Kaggle account
# the first time (upload kaggle.json, or use kagglehub's built-in login)
path = kagglehub.dataset_download("subhajournal/phishingemails")
print("Dataset downloaded to:", path)

import os
print(os.listdir(path))

# Load it — adjust filename if listed differently above
email_file = os.path.join(path, "Phishing_Email.csv")
df_email_raw = pd.read_csv(email_file)

print("\n=== Correct Email Dataset ===")
print("Shape:", df_email_raw.shape)
print(df_email_raw.head())
print("\nLabel distribution:\n", df_email_raw['Email Type'].value_counts())

# -----------------------------------------------------------
# 2b. Clean the email dataset
# -----------------------------------------------------------
df_email_clean = df_email_raw.copy()

# Drop unnamed index columns if present
df_email_clean = df_email_clean.loc[:, ~df_email_clean.columns.str.contains('^Unnamed')]

# Rename to consistent schema
df_email_clean = df_email_clean.rename(columns={
    'Email Text': 'text',
    'Email Type': 'label_raw'
})

# Drop missing/empty text rows
df_email_clean = df_email_clean.dropna(subset=['text'])
df_email_clean = df_email_clean[df_email_clean['text'].str.strip() != '']

# Drop exact duplicate emails
before = len(df_email_clean)
df_email_clean = df_email_clean.drop_duplicates(subset=['text'])
print(f"\nEmail dataset: dropped {before - len(df_email_clean)} duplicate rows")

# Encode label: Phishing Email -> 1, Safe Email -> 0
df_email_clean['label'] = df_email_clean['label_raw'].map({
    'Phishing Email': 1,
    'Safe Email': 0
})

# Sanity check — make sure mapping worked (no NaNs introduced)
assert df_email_clean['label'].isnull().sum() == 0, "Unmapped label values found — check unique values in label_raw"

print("\nFinal email dataset shape:", df_email_clean.shape)
print("Final label distribution:\n", df_email_clean['label'].value_counts(normalize=True))

# -----------------------------------------------------------
# 2c. Clean the URL dataset
# -----------------------------------------------------------
df_url_clean = df_url.copy()

# Drop duplicates based on URL
before = len(df_url_clean)
df_url_clean = df_url_clean.drop_duplicates(subset=['URL'])
print(f"\nURL dataset: dropped {before - len(df_url_clean)} duplicate rows")

# Confirm no missing values remain
missing = df_url_clean.isnull().sum()
print("\nRemaining missing values (URL dataset):\n", missing[missing > 0] if missing.sum() > 0 else "None")

# Confirm label distribution
print("\nURL label distribution:\n", df_url_clean['label'].value_counts(normalize=True))

print("\n=== Clean dataset shapes ===")
print("URL dataset:", df_url_clean.shape)
print("Email dataset:", df_email_clean.shape)

In [ ]:
# =========================================================
# PIECE 3: Feature Engineering
# =========================================================
import re
import math
from collections import Counter

# -----------------------------------------------------------
# 3a. URL DATASET — extend PhiUSIIL's existing features
# -----------------------------------------------------------
df_url_feat = df_url_clean.copy()

def url_entropy(url):
    """Shannon entropy of the URL string — obfuscated/randomized URLs score higher."""
    if not url:
        return 0
    counts = Counter(url)
    length = len(url)
    return -sum((c / length) * math.log2(c / length) for c in counts.values())

SUSPICIOUS_KEYWORDS = ['login', 'secure', 'account', 'verify', 'update',
                        'confirm', 'bank', 'signin', 'security', 'ebay',
                        'paypal', 'webscr']

df_url_feat['URLEntropy'] = df_url_feat['URL'].apply(url_entropy)
df_url_feat['HasSuspiciousKeyword'] = df_url_feat['URL'].str.lower().apply(
    lambda u: int(any(kw in u for kw in SUSPICIOUS_KEYWORDS))
)
df_url_feat['DigitToLetterRatio'] = df_url_feat['NoOfDegitsInURL'] / (df_url_feat['NoOfLettersInURL'] + 1)

# Drop non-numeric / identifier columns not usable as model features
url_drop_cols = ['URL', 'Domain', 'TLD', 'Title']
X_url = df_url_feat.drop(columns=url_drop_cols + ['label'])
y_url = df_url_feat['label']

print("=== URL feature matrix ===")
print("X_url shape:", X_url.shape)
print("Feature dtypes:\n", X_url.dtypes.value_counts())
print("Any non-numeric columns left?", list(X_url.select_dtypes(exclude=[np.number]).columns))

# -----------------------------------------------------------
# 3b. EMAIL DATASET — derive structural/behavioral-proxy
#     features from raw text (this dataset has none yet)
# -----------------------------------------------------------
df_email_feat = df_email_clean.copy()

URGENCY_KEYWORDS = ['urgent', 'immediately', 'verify', 'suspend', 'click here',
                     'act now', 'limited time', 'confirm', 'password',
                     'security alert', 'account will be', 'update your',
                     'winner', 'congratulations', 'free', 'click below']

def count_keywords(text, keywords):
    text_lower = str(text).lower()
    return sum(text_lower.count(kw) for kw in keywords)

def count_links(text):
    return len(re.findall(r'https?://\S+|www\.\S+', str(text)))

def count_html_tags(text):
    return len(re.findall(r'<[^>]+>', str(text)))

def uppercase_word_ratio(text):
    words = str(text).split()
    if not words:
        return 0
    upper = sum(1 for w in words if w.isupper() and len(w) > 1)
    return upper / len(words)

df_email_feat['text_length'] = df_email_feat['text'].str.len()
df_email_feat['word_count'] = df_email_feat['text'].str.split().str.len()
df_email_feat['num_links'] = df_email_feat['text'].apply(count_links)
df_email_feat['num_exclamations'] = df_email_feat['text'].str.count('!')
df_email_feat['num_digits'] = df_email_feat['text'].apply(lambda t: sum(c.isdigit() for c in str(t)))
df_email_feat['urgency_keyword_count'] = df_email_feat['text'].apply(
    lambda t: count_keywords(t, URGENCY_KEYWORDS)
)
df_email_feat['num_html_tags'] = df_email_feat['text'].apply(count_html_tags)
df_email_feat['uppercase_word_ratio'] = df_email_feat['text'].apply(uppercase_word_ratio)
df_email_feat['avg_word_length'] = df_email_feat['text_length'] / (df_email_feat['word_count'] + 1)
df_email_feat['url_density'] = df_email_feat['num_links'] / (df_email_feat['word_count'] + 1)

email_feature_cols = ['text_length', 'word_count', 'num_links', 'num_exclamations',
                       'num_digits', 'urgency_keyword_count', 'num_html_tags',
                       'uppercase_word_ratio', 'avg_word_length', 'url_density']

X_email = df_email_feat[email_feature_cols]
y_email = df_email_feat['label']

print("\n=== Email feature matrix ===")
print("X_email shape:", X_email.shape)
print(X_email.describe())

# -----------------------------------------------------------
# 3c. Save feature-engineered datasets (this is your
#     "feature engineered dataset" deliverable)
# -----------------------------------------------------------
OUT_DIR = '/content/drive/MyDrive/phishing_project/'

X_url.assign(label=y_url).to_csv(OUT_DIR + 'url_features_engineered.csv', index=False)
X_email.assign(label=y_email).to_csv(OUT_DIR + 'email_features_engineered.csv', index=False)

print("\nSaved feature-engineered datasets to Drive.")

In [ ]:
# =========================================================
# PIECE 3b: TF-IDF / Bag-of-Words for Email Text
# =========================================================
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix
import pickle

# -----------------------------------------------------------
# 3b-1. Basic text cleaning before vectorizing
#       (light touch — keep punctuation patterns like "!!!"
#       and links out, since num_exclamations/num_links
#       already capture those signals separately)
# -----------------------------------------------------------
def clean_for_tfidf(text):
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' URLTOKEN ', text)   # normalize links
    text = re.sub(r'<[^>]+>', ' ', text)                          # strip HTML tags
    text = re.sub(r'[^a-z\s]', ' ', text)                         # keep letters only
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_email_feat['text_clean'] = df_email_feat['text'].apply(clean_for_tfidf)

# -----------------------------------------------------------
# 3b-2. Fit TF-IDF (unigrams + bigrams, capped vocabulary
#       so it stays manageable for tree-based models)
# -----------------------------------------------------------
tfidf = TfidfVectorizer(
    max_features=500,      # cap vocab size — keep this modest for RF/boosting speed
    ngram_range=(1, 2),    # unigrams + bigrams (bigrams catch phrases like "click here")
    stop_words='english',
    min_df=5,               # ignore terms in fewer than 5 emails (noise reduction)
    max_df=0.9               # ignore terms in >90% of emails (too generic to help)
)

X_tfidf = tfidf.fit_transform(df_email_feat['text_clean'])
print("TF-IDF matrix shape:", X_tfidf.shape)
print("Sample vocabulary:", tfidf.get_feature_names_out()[:20])

# -----------------------------------------------------------
# 3b-3. Combine handcrafted features + TF-IDF into one matrix
# -----------------------------------------------------------
X_email_handcrafted_sparse = csr_matrix(X_email.values)  # X_email from Piece 3
X_email_combined = hstack([X_email_handcrafted_sparse, X_tfidf]).tocsr()

combined_feature_names = list(X_email.columns) + [f"tfidf_{w}" for w in tfidf.get_feature_names_out()]

print("\nCombined email feature matrix shape:", X_email_combined.shape)
print(f"({len(email_feature_cols)} handcrafted + {X_tfidf.shape[1]} TF-IDF = {X_email_combined.shape[1]} total)")

# -----------------------------------------------------------
# 3b-4. Save the combined matrix + vectorizer
#       (saved as sparse .npz — CSV would be huge/wasteful
#       for a mostly-zero TF-IDF matrix)
# -----------------------------------------------------------
from scipy.sparse import save_npz

save_npz(OUT_DIR + 'email_features_combined.npz', X_email_combined)
np.save(OUT_DIR + 'email_labels.npy', y_email.values)

with open(OUT_DIR + 'tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

with open(OUT_DIR + 'combined_feature_names.pkl', 'wb') as f:
    pickle.dump(combined_feature_names, f)

print("\nSaved combined feature matrix, labels, vectorizer, and feature names to Drive.")

In [ ]:
# =========================================================
# PIECE 4: Train Base Models — RF, XGBoost, LightGBM, CatBoost
# =========================================================
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import pickle
import time

MODEL_DIR = OUT_DIR + 'models/'
os.makedirs(MODEL_DIR, exist_ok=True)

# -----------------------------------------------------------
# 4a. Train/test split — stratified, same seed everywhere
#     so results are directly comparable across models later
# -----------------------------------------------------------
X_url_train, X_url_test, y_url_train, y_url_test = train_test_split(
    X_url, y_url, test_size=0.2, stratify=y_url, random_state=RANDOM_STATE
)

X_email_train, X_email_test, y_email_train, y_email_test = train_test_split(
    X_email_combined, y_email, test_size=0.2, stratify=y_email, random_state=RANDOM_STATE
)

print("URL train/test:", X_url_train.shape, X_url_test.shape)
print("Email train/test:", X_email_train.shape, X_email_test.shape)

# -----------------------------------------------------------
# 4b. Define base models (default-ish params — tuning comes
#     in Piece 5 with Optuna, this is the baseline pass)
# -----------------------------------------------------------
def get_base_models():
    return {
        'RandomForest': RandomForestClassifier(
            n_estimators=200, max_depth=20, random_state=RANDOM_STATE, n_jobs=-1
        ),
        'XGBoost': XGBClassifier(
            n_estimators=200, max_depth=6, learning_rate=0.1,
            eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1
        ),
        'LightGBM': LGBMClassifier(
            n_estimators=200, max_depth=6, learning_rate=0.1,
            random_state=RANDOM_STATE, n_jobs=-1, verbose=-1
        ),
        'CatBoost': CatBoostClassifier(
            iterations=200, depth=6, learning_rate=0.1,
            random_state=RANDOM_STATE, verbose=0
        )
    }

# -----------------------------------------------------------
# 4c. Generic training loop — trains all 4 models on a given
#     dataset, stores fitted models + predictions + timing
# -----------------------------------------------------------
def train_all_models(X_train, y_train, X_test, y_test, dataset_name):
    results = {}
    models = get_base_models()

    for name, model in models.items():
        print(f"\nTraining {name} on {dataset_name}...")
        start = time.time()
        model.fit(X_train, y_train)
        elapsed = time.time() - start

        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]

        results[name] = {
            'model': model,
            'y_pred': y_pred,
            'y_proba': y_proba,
            'train_time_sec': elapsed
        }
        print(f"  Done in {elapsed:.1f}s | Test accuracy: {(y_pred == y_test).mean():.4f}")

    return results

# -----------------------------------------------------------
# 4d. Train on both datasets
# -----------------------------------------------------------
print("=" * 60)
print("TRAINING ON URL DATASET")
print("=" * 60)
url_results = train_all_models(X_url_train, y_url_train, X_url_test, y_url_test, "URL")

print("\n" + "=" * 60)
print("TRAINING ON EMAIL DATASET")
print("=" * 60)
email_results = train_all_models(X_email_train, y_email_train, X_email_test, y_email_test, "Email")

# -----------------------------------------------------------
# 4e. Save trained models + train/test splits for later pieces
#     (hyperparameter tuning, stacking, evaluation)
# -----------------------------------------------------------
for dataset_name, results in [('url', url_results), ('email', email_results)]:
    for model_name, res in results.items():
        with open(f"{MODEL_DIR}{dataset_name}_{model_name}_baseline.pkl", 'wb') as f:
            pickle.dump(res['model'], f)

# Save splits so every later piece uses the exact same partitions
with open(MODEL_DIR + 'splits.pkl', 'wb') as f:
    pickle.dump({
        'X_url_train': X_url_train, 'X_url_test': X_url_test,
        'y_url_train': y_url_train, 'y_url_test': y_url_test,
        'X_email_train': X_email_train, 'X_email_test': X_email_test,
        'y_email_train': y_email_train, 'y_email_test': y_email_test,
    }, f)

print("\nAll baseline models and splits saved to Drive.")

# -----------------------------------------------------------
# 4f. Quick baseline comparison table (this is a first pass —
#     the full accuracy comparison table deliverable comes
#     in the evaluation piece with all metrics together)
# -----------------------------------------------------------
from sklearn.metrics import accuracy_score

summary_rows = []
for dataset_name, results, y_test in [('URL', url_results, y_url_test), ('Email', email_results, y_email_test)]:
    for model_name, res in results.items():
        summary_rows.append({
            'Dataset': dataset_name,
            'Model': model_name,
            'Accuracy': accuracy_score(y_test, res['y_pred']),
            'Train Time (s)': round(res['train_time_sec'], 1)
        })

baseline_summary = pd.DataFrame(summary_rows).sort_values(['Dataset', 'Accuracy'], ascending=[True, False])
print("\n=== Baseline Model Comparison ===")
print(baseline_summary.to_string(index=False))

In [ ]:
# =========================================================
# PIECE 4b: Diagnose Data Leakage in the URL Dataset
# =========================================================

# -----------------------------------------------------------
# 4b-1. Check feature importances from the trained RF —
#       a leaking feature usually dominates everything else
# -----------------------------------------------------------
rf_url = url_results['RandomForest']['model']

importances = pd.Series(rf_url.feature_importances_, index=X_url.columns)
importances = importances.sort_values(ascending=False)

print("=== Top 15 most important features (URL, RandomForest) ===")
print(importances.head(15))

# -----------------------------------------------------------
# 4b-2. Check raw correlation of each feature with the label
#       (only works cleanly for numeric columns)
# -----------------------------------------------------------
numeric_X = X_url.select_dtypes(include=[np.number])
corr_with_label = numeric_X.corrwith(y_url).abs().sort_values(ascending=False)

print("\n=== Top 15 features most correlated with label ===")
print(corr_with_label.head(15))

In [ ]:
# =========================================================
# PIECE 4c: Drop Leaky Features and Retrain
# =========================================================

# Adjust this list based on what 4b actually shows —
# these three are the most likely culprits in PhiUSIIL
LEAKY_COLS = ['URLSimilarityIndex', 'TLDLegitimateProb',
              'DomainTitleMatchScore', 'URLTitleMatchScore']

leaky_present = [c for c in LEAKY_COLS if c in X_url.columns]
print("Dropping suspected leaky columns:", leaky_present)

X_url_safe = X_url.drop(columns=leaky_present)

X_url_train2, X_url_test2, y_url_train2, y_url_test2 = train_test_split(
    X_url_safe, y_url, test_size=0.2, stratify=y_url, random_state=RANDOM_STATE
)

print("\nRetraining RandomForest without leaky columns...")
rf_check = RandomForestClassifier(n_estimators=200, max_depth=20, random_state=RANDOM_STATE, n_jobs=-1)
rf_check.fit(X_url_train2, y_url_train2)
acc_check = (rf_check.predict(X_url_test2) == y_url_test2).mean()
print(f"Test accuracy without leaky columns: {acc_check:.4f}")

In [ ]:
# =========================================================
# PIECE 4d: Split URL Features into Lexical-Only vs Full-Page
# =========================================================

# -----------------------------------------------------------
# Lexical/structural URL-STRING features only — everything
# computable instantly from the URL text alone, before ever
# visiting the site. This is what a real-time, pre-click
# filter would realistically have access to.
# -----------------------------------------------------------
URL_LEXICAL_COLS = [
    'URLLength', 'DomainLength', 'IsDomainIP', 'TLDLength',
    'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar',
    'ObfuscationRatio', 'NoOfLettersInURL', 'LetterRatioInURL',
    'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL',
    'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL',
    'SpacialCharRatioInURL', 'URLEntropy', 'HasSuspiciousKeyword',
    'DigitToLetterRatio'
]

# Everything else in X_url_safe is page-content/HTML-based —
# requires actually crawling/rendering the page
url_lexical_present = [c for c in URL_LEXICAL_COLS if c in X_url_safe.columns]
url_content_cols = [c for c in X_url_safe.columns if c not in url_lexical_present]

print("Lexical (URL-string-only) features:", len(url_lexical_present))
print("Page-content (crawled) features:", len(url_content_cols))
print("\nContent columns:", url_content_cols)

X_url_lexical = X_url_safe[url_lexical_present]
X_url_full = X_url_safe  # lexical + page-content combined

# -----------------------------------------------------------
# Retrain RF on lexical-only features to see the honest,
# real-time-realistic accuracy
# -----------------------------------------------------------
X_lex_train, X_lex_test, y_lex_train, y_lex_test = train_test_split(
    X_url_lexical, y_url, test_size=0.2, stratify=y_url, random_state=RANDOM_STATE
)

rf_lexical = RandomForestClassifier(n_estimators=200, max_depth=20, random_state=RANDOM_STATE, n_jobs=-1)
rf_lexical.fit(X_lex_train, y_lex_train)
acc_lexical = (rf_lexical.predict(X_lex_test) == y_lex_test).mean()
print(f"\nTest accuracy — lexical-only features: {acc_lexical:.4f}")

In [ ]:
# =========================================================
# PIECE 4 (FINAL): Train Base Models Across All 3 Feature Sets
# =========================================================

# -----------------------------------------------------------
# 4-final-a. Define the three feature sets consistently
# -----------------------------------------------------------
datasets = {
    'URL_Lexical': (X_url_lexical, y_url),
    'URL_Full':    (X_url_full, y_url),
    'Email':       (X_email_combined, y_email),
}

splits = {}
for name, (X, y) in datasets.items():
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
    )
    splits[name] = (X_train, X_test, y_train, y_test)
    print(f"{name}: train {X_train.shape}, test {X_test.shape}")

# -----------------------------------------------------------
# 4-final-b. Train all 4 models on all 3 feature sets
# -----------------------------------------------------------
all_results = {}
for name, (X_train, X_test, y_train, y_test) in splits.items():
    print("\n" + "=" * 60)
    print(f"TRAINING ON {name}")
    print("=" * 60)
    all_results[name] = train_all_models(X_train, y_train, X_test, y_test, name)

# -----------------------------------------------------------
# 4-final-c. Save everything — models, splits, summary
# -----------------------------------------------------------
for dataset_name, results in all_results.items():
    for model_name, res in results.items():
        with open(f"{MODEL_DIR}{dataset_name}_{model_name}.pkl", 'wb') as f:
            pickle.dump(res['model'], f)

with open(MODEL_DIR + 'splits_final.pkl', 'wb') as f:
    pickle.dump(splits, f)

# -----------------------------------------------------------
# 4-final-d. Full comparison table across all 3 feature sets
# -----------------------------------------------------------
summary_rows = []
for dataset_name, results in all_results.items():
    _, _, _, y_test = splits[dataset_name]
    for model_name, res in results.items():
        summary_rows.append({
            'Feature Set': dataset_name,
            'Model': model_name,
            'Accuracy': accuracy_score(y_test, res['y_pred']),
            'Train Time (s)': round(res['train_time_sec'], 1)
        })

full_summary = pd.DataFrame(summary_rows).sort_values(['Feature Set', 'Accuracy'], ascending=[True, False])
print("\n=== Full Baseline Comparison (3 feature sets x 4 models) ===")
print(full_summary.to_string(index=False))

full_summary.to_csv(OUT_DIR + 'baseline_comparison_table.csv', index=False)
print("\nSaved comparison table to Drive.")

In [ ]:
# =========================================================
# PIECE 5 (GPU): Hyperparameter Tuning with Optuna
# =========================================================
import optuna
from sklearn.model_selection import StratifiedKFold, cross_val_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

# -----------------------------------------------------------
# 5a-0. Confirm GPU is actually available before relying on it
# -----------------------------------------------------------
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

import torch  # just for a quick reliable GPU check, not used for training
GPU_AVAILABLE = torch.cuda.is_available()
print("GPU available:", GPU_AVAILABLE)

# -----------------------------------------------------------
# 5a. Feature sets we're tuning — URL_Full skipped (saturated)
# -----------------------------------------------------------
tuning_targets = {
    'URL_Lexical': splits['URL_Lexical'],
    'Email': splits['Email'],
}

CV = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
N_TRIALS = 30

# -----------------------------------------------------------
# 5b. Objective functions — GPU configs added for XGB/LGBM/CatBoost.
#     RandomForest is unchanged (CPU-only, no GPU support in sklearn).
# -----------------------------------------------------------
def objective_rf(trial, X, y):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 400),
        'max_depth': trial.suggest_int('max_depth', 5, 30),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
        'random_state': RANDOM_STATE, 'n_jobs': -1
    }
    model = RandomForestClassifier(**params)
    return cross_val_score(model, X, y, cv=CV, scoring='f1', n_jobs=-1).mean()

def objective_xgb(trial, X, y):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 400),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'eval_metric': 'logloss', 'random_state': RANDOM_STATE,
    }
    if GPU_AVAILABLE:
        params['device'] = 'cuda'
        params['tree_method'] = 'hist'
    else:
        params['n_jobs'] = -1
    model = XGBClassifier(**params)
    # cross_val_score with n_jobs=-1 would spawn multiple processes fighting
    # over one GPU — force sequential CV folds when GPU is active
    return cross_val_score(model, X, y, cv=CV, scoring='f1',
                            n_jobs=1 if GPU_AVAILABLE else -1).mean()

def objective_lgbm(trial, X, y):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 400),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 127),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'random_state': RANDOM_STATE, 'verbose': -1,
    }
    if GPU_AVAILABLE:
        params['device'] = 'gpu'
    else:
        params['n_jobs'] = -1
    model = LGBMClassifier(**params)
    return cross_val_score(model, X, y, cv=CV, scoring='f1',
                            n_jobs=1 if GPU_AVAILABLE else -1).mean()

def objective_catboost(trial, X, y):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 400),
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        'random_state': RANDOM_STATE, 'verbose': 0,
    }
    if GPU_AVAILABLE:
        params['task_type'] = 'GPU'
        params['devices'] = '0'
    model = CatBoostClassifier(**params)
    return cross_val_score(model, X, y, cv=CV, scoring='f1',
                            n_jobs=1 if GPU_AVAILABLE else -1).mean()

OBJECTIVES = {
    'RandomForest': objective_rf,
    'XGBoost': objective_xgb,
    'LightGBM': objective_lgbm,
    'CatBoost': objective_catboost,
}

# -----------------------------------------------------------
# 5c. Run tuning — one Optuna study per (feature set, model)
# -----------------------------------------------------------
best_params = {}
for feat_name, (X_train, X_test, y_train, y_test) in tuning_targets.items():
    best_params[feat_name] = {}
    for model_name, obj_fn in OBJECTIVES.items():
        print(f"\nTuning {model_name} on {feat_name} ({N_TRIALS} trials)...")
        study = optuna.create_study(direction='maximize')
        study.optimize(lambda trial: obj_fn(trial, X_train, y_train),
                        n_trials=N_TRIALS, show_progress_bar=False)
        print(f"  Best F1 (CV): {study.best_value:.4f}")
        print(f"  Best params: {study.best_params}")
        best_params[feat_name][model_name] = study.best_params

# -----------------------------------------------------------
# 5d. Retrain final models on full train set with best params
# -----------------------------------------------------------
MODEL_CLASSES = {
    'RandomForest': RandomForestClassifier,
    'XGBoost': XGBClassifier,
    'LightGBM': LGBMClassifier,
    'CatBoost': CatBoostClassifier,
}

def fixed_params_for(model_name):
    if model_name == 'RandomForest':
        return {'random_state': RANDOM_STATE, 'n_jobs': -1}
    if model_name == 'XGBoost':
        p = {'eval_metric': 'logloss', 'random_state': RANDOM_STATE}
        p.update({'device': 'cuda', 'tree_method': 'hist'} if GPU_AVAILABLE else {'n_jobs': -1})
        return p
    if model_name == 'LightGBM':
        p = {'random_state': RANDOM_STATE, 'verbose': -1}
        p.update({'device': 'gpu'} if GPU_AVAILABLE else {'n_jobs': -1})
        return p
    if model_name == 'CatBoost':
        p = {'random_state': RANDOM_STATE, 'verbose': 0}
        p.update({'task_type': 'GPU', 'devices': '0'} if GPU_AVAILABLE else {})
        return p

tuned_results = {}
tuned_models = {}
for feat_name, (X_train, X_test, y_train, y_test) in tuning_targets.items():
    tuned_results[feat_name] = {}
    tuned_models[feat_name] = {}
    for model_name in OBJECTIVES.keys():
        params = {**best_params[feat_name][model_name], **fixed_params_for(model_name)}
        model = MODEL_CLASSES[model_name](**params)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]
        acc = accuracy_score(y_test, y_pred)

        tuned_results[feat_name][model_name] = {
            'model': model, 'y_pred': y_pred, 'y_proba': y_proba, 'accuracy': acc
        }
        tuned_models[feat_name][model_name] = model
        print(f"{feat_name} / {model_name}: tuned test accuracy = {acc:.4f}")

# -----------------------------------------------------------
# 5e. Save tuned models + best params
# -----------------------------------------------------------
for feat_name, models in tuned_models.items():
    for model_name, model in models.items():
        with open(f"{MODEL_DIR}{feat_name}_{model_name}_tuned.pkl", 'wb') as f:
            pickle.dump(model, f)

with open(MODEL_DIR + 'best_params.pkl', 'wb') as f:
    pickle.dump(best_params, f)

print("\nAll tuned models and best params saved to Drive.")

# -----------------------------------------------------------
# 5f. Before/after comparison table (baseline vs tuned)
# -----------------------------------------------------------
comparison_rows = []
for feat_name in tuning_targets.keys():
    _, _, _, y_test = tuning_targets[feat_name]
    for model_name in OBJECTIVES.keys():
        baseline_acc = accuracy_score(y_test, all_results[feat_name][model_name]['y_pred'])
        tuned_acc = tuned_results[feat_name][model_name]['accuracy']
        comparison_rows.append({
            'Feature Set': feat_name,
            'Model': model_name,
            'Baseline Accuracy': baseline_acc,
            'Tuned Accuracy': tuned_acc,
            'Improvement': tuned_acc - baseline_acc
        })

tuning_comparison = pd.DataFrame(comparison_rows)
print("\n=== Baseline vs Tuned Comparison ===")
print(tuning_comparison.to_string(index=False))
tuning_comparison.to_csv(OUT_DIR + 'tuning_comparison_table.csv', index=False)

In [ ]:
# =========================================================
# TEMPORARY: Quick 1-2 trial test run (standalone, no edits to Piece 5)
# =========================================================
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

TEST_N_TRIALS = 2

X_train_test, X_test_test, y_train_test, y_test_test = splits['URL_Lexical']

print("Quick test: LightGBM on URL_Lexical,", TEST_N_TRIALS, "trials")
study_test = optuna.create_study(direction='maximize')
study_test.optimize(lambda trial: objective_lgbm(trial, X_train_test, y_train_test),
                     n_trials=TEST_N_TRIALS, show_progress_bar=False)

print("Best F1 (CV):", study_test.best_value)
print("Best params:", study_test.best_params)
print("\nIf this ran without errors, the GPU config and pipeline are working.")

In [ ]:
print(best_params['URL_Lexical'].keys())
print(best_params.get('Email', {}).keys())

In [ ]:
print(list(OBJECTIVES.keys()))
print(list(tuning_targets.keys()))
print("N_TRIALS =", N_TRIALS)

In [ ]:
# =========================================================
# PATCH: Fallback to baseline defaults for any untuned model
# =========================================================
BASELINE_DEFAULTS = {
    'RandomForest': {'n_estimators': 200, 'max_depth': 20},
    'XGBoost': {'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.1},
    'LightGBM': {'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.1},
    'CatBoost': {'iterations': 200, 'depth': 6, 'learning_rate': 0.1},
}

def get_stacking_estimators(feat_name):
    tuned = best_params.get(feat_name, {})

    def params_for(model_name):
        return tuned[model_name] if model_name in tuned else BASELINE_DEFAULTS[model_name]

    estimators = [
        ('rf', RandomForestClassifier(**params_for('RandomForest'),
                                       random_state=RANDOM_STATE, n_jobs=-1)),
        ('xgb', XGBClassifier(**params_for('XGBoost'),
                               eval_metric='logloss', random_state=RANDOM_STATE)),
        ('lgbm', LGBMClassifier(**params_for('LightGBM'),
                                 random_state=RANDOM_STATE, verbose=-1)),
        ('catboost', CatBoostClassifier(**params_for('CatBoost'),
                                         random_state=RANDOM_STATE, verbose=0)),
    ]

    used_defaults = [m for m in BASELINE_DEFAULTS if m not in tuned]
    if used_defaults:
        print(f"  Note: using baseline (untuned) params for {used_defaults} on {feat_name}")

    return estimators

In [ ]:
# =========================================================
# PIECE 6: Stacking Ensemble (Meta-Learner over 4 Base Models)
# =========================================================
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
import time

# -----------------------------------------------------------
# 6a. Build the base estimator list per feature set — use
#     TUNED hyperparameters where available (URL_Lexical, Email),
#     fall back to baseline defaults for URL_Full (not tuned,
#     already saturated near 100%)
# -----------------------------------------------------------
def get_stacking_estimators(feat_name):
    if feat_name in best_params:
        # Use tuned hyperparameters
        params = best_params[feat_name]
        estimators = [
            ('rf', RandomForestClassifier(**params['RandomForest'],
                                           random_state=RANDOM_STATE, n_jobs=-1)),
            ('xgb', XGBClassifier(**params['XGBoost'],
                                   eval_metric='logloss', random_state=RANDOM_STATE)),
            ('lgbm', LGBMClassifier(**params['LightGBM'],
                                     random_state=RANDOM_STATE, verbose=-1)),
            ('catboost', CatBoostClassifier(**params['CatBoost'],
                                             random_state=RANDOM_STATE, verbose=0)),
        ]
    else:
        # Fallback: baseline defaults (used for URL_Full only)
        estimators = [
            ('rf', RandomForestClassifier(n_estimators=200, max_depth=20,
                                           random_state=RANDOM_STATE, n_jobs=-1)),
            ('xgb', XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                                   eval_metric='logloss', random_state=RANDOM_STATE)),
            ('lgbm', LGBMClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                                     random_state=RANDOM_STATE, verbose=-1)),
            ('catboost', CatBoostClassifier(iterations=200, depth=6, learning_rate=0.1,
                                             random_state=RANDOM_STATE, verbose=0)),
        ]
    return estimators

# -----------------------------------------------------------
# 6b. Build and train a stacking ensemble for one feature set
# -----------------------------------------------------------
def train_stacking_ensemble(feat_name, X_train, y_train, X_test, y_test, cv=3):
    print(f"\nBuilding stacking ensemble for {feat_name}...")
    estimators = get_stacking_estimators(feat_name)

    meta_learner = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)

    stack = StackingClassifier(
        estimators=estimators,
        final_estimator=meta_learner,
        cv=cv,                  # generates proper out-of-fold predictions
        stack_method='predict_proba',
        n_jobs=1,                # keep sequential — GPU models fight over one GPU otherwise
        passthrough=False        # meta-learner sees ONLY base model outputs, not raw features
    )

    start = time.time()
    stack.fit(X_train, y_train)
    elapsed = time.time() - start

    y_pred = stack.predict(X_test)
    y_proba = stack.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print(f"  Trained in {elapsed:.1f}s")
    print(f"  Stacking ensemble — Accuracy: {acc:.4f} | F1: {f1:.4f}")

    return {
        'model': stack,
        'y_pred': y_pred,
        'y_proba': y_proba,
        'accuracy': acc,
        'f1': f1,
        'train_time_sec': elapsed
    }

# =========================================================
# PATCH: Fallback to baseline defaults for any untuned model
# =========================================================
BASELINE_DEFAULTS = {
    'RandomForest': {'n_estimators': 200, 'max_depth': 20},
    'XGBoost': {'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.1},
    'LightGBM': {'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.1},
    'CatBoost': {'iterations': 200, 'depth': 6, 'learning_rate': 0.1},
}

def get_stacking_estimators(feat_name):
    tuned = best_params.get(feat_name, {})

    def params_for(model_name):
        return tuned[model_name] if model_name in tuned else BASELINE_DEFAULTS[model_name]

    estimators = [
        ('rf', RandomForestClassifier(**params_for('RandomForest'),
                                       random_state=RANDOM_STATE, n_jobs=-1)),
        ('xgb', XGBClassifier(**params_for('XGBoost'),
                               eval_metric='logloss', random_state=RANDOM_STATE)),
        ('lgbm', LGBMClassifier(**params_for('LightGBM'),
                                 random_state=RANDOM_STATE, verbose=-1)),
        ('catboost', CatBoostClassifier(**params_for('CatBoost'),
                                         random_state=RANDOM_STATE, verbose=0)),
    ]

    used_defaults = [m for m in BASELINE_DEFAULTS if m not in tuned]
    if used_defaults:
        print(f"  Note: using baseline (untuned) params for {used_defaults} on {feat_name}")

    return estimators

# -----------------------------------------------------------
# 6c. Run stacking for all three feature sets
# -----------------------------------------------------------
stacking_results = {}
for feat_name in ['URL_Lexical', 'URL_Full', 'Email']:
    X_train, X_test, y_train, y_test = splits[feat_name]
    stacking_results[feat_name] = train_stacking_ensemble(
        feat_name, X_train, y_train, X_test, y_test
    )

# -----------------------------------------------------------
# 6d. Save stacking models
# -----------------------------------------------------------
for feat_name, res in stacking_results.items():
    with open(f"{MODEL_DIR}{feat_name}_StackingEnsemble.pkl", 'wb') as f:
        pickle.dump(res['model'], f)

print("\nStacking ensembles saved to Drive.")

# =========================================================
# GUARD: Define tuned_results as fallback only if it doesn't exist
# =========================================================
if 'tuned_results' not in dir():
    tuned_results = all_results
    print("tuned_results not found — using all_results (baseline) as fallback")

# -----------------------------------------------------------
# 6e. Compare stacking vs best individual model per feature set
# -----------------------------------------------------------
comparison_rows = []
for feat_name in ['URL_Lexical', 'URL_Full', 'Email']:
    _, _, _, y_test = splits[feat_name]

    # Best individual tuned model (or baseline for URL_Full)
    individual_source = tuned_results if feat_name in tuned_results else all_results
    best_individual_name = max(
        individual_source[feat_name],
        key=lambda m: accuracy_score(y_test, individual_source[feat_name][m]['y_pred'])
    )
    best_individual_acc = accuracy_score(
        y_test, individual_source[feat_name][best_individual_name]['y_pred']
    )
    best_individual_f1 = f1_score(
        y_test, individual_source[feat_name][best_individual_name]['y_pred']
    )

    comparison_rows.append({
        'Feature Set': feat_name,
        'Best Individual Model': best_individual_name,
        'Individual Accuracy': best_individual_acc,
        'Individual F1': best_individual_f1,
        'Stacking Accuracy': stacking_results[feat_name]['accuracy'],
        'Stacking F1': stacking_results[feat_name]['f1'],
        'Accuracy Gain': stacking_results[feat_name]['accuracy'] - best_individual_acc,
        'F1 Gain': stacking_results[feat_name]['f1'] - best_individual_f1,
    })

stacking_comparison = pd.DataFrame(comparison_rows)
print("\n=== Stacking vs Best Individual Model ===")
print(stacking_comparison.to_string(index=False))
stacking_comparison.to_csv(OUT_DIR + 'stacking_comparison_table.csv', index=False)

In [ ]:
# =========================================================
# PIECE 7: SHAP Explainability Layer
# =========================================================
import shap

# -----------------------------------------------------------
# 7a. Pick which model to explain per feature set — use the
#     stacking ensemble's best base learner (SHAP doesn't
#     directly support StackingClassifier's meta-layer well,
#     so we explain the strongest underlying tree model instead)
# -----------------------------------------------------------
# For URL_Lexical and URL_Full: explain LightGBM (best individual)
# For Email: explain XGBoost (best individual)
explain_targets = {
    'URL_Lexical': ('LightGBM', all_results['URL_Lexical']['LightGBM']['model'], X_url_lexical),
    'Email': ('XGBoost', all_results['Email']['XGBoost']['model'], None),  # handled separately below (sparse)
}

# -----------------------------------------------------------
# 7b. SHAP for URL_Lexical (dense features — straightforward)
# -----------------------------------------------------------
print("Computing SHAP values for URL_Lexical (LightGBM)...")
model_name, model, X_full = explain_targets['URL_Lexical']
_, X_test_url, _, y_test_url = splits['URL_Lexical']

# Use a sample for speed — SHAP on 47K rows is slow; 1000 is plenty for insight
sample_idx = np.random.RandomState(RANDOM_STATE).choice(len(X_test_url), size=1000, replace=False)
X_sample_url = X_test_url.iloc[sample_idx]

explainer_url = shap.TreeExplainer(model)
shap_values_url = explainer_url.shap_values(X_sample_url)

# For binary classification, some models return a list [class0, class1] — normalize
if isinstance(shap_values_url, list):
    shap_values_url = shap_values_url[1]

print("Generating summary plot (URL_Lexical)...")
shap.summary_plot(shap_values_url, X_sample_url, show=False)
plt.title("SHAP Summary — URL Lexical Features (LightGBM)")
plt.tight_layout()
plt.savefig(OUT_DIR + 'shap_summary_url_lexical.png', dpi=150, bbox_inches='tight')
plt.show()

# -----------------------------------------------------------
# 7c. SHAP for Email (sparse combined features — needs care)
# -----------------------------------------------------------
print("\nComputing SHAP values for Email (XGBoost)...")
_, X_test_email, _, y_test_email = splits['Email']
email_model = all_results['Email']['XGBoost']['model']

# Sample for speed
sample_idx_email = np.random.RandomState(RANDOM_STATE).choice(
    X_test_email.shape[0], size=min(1000, X_test_email.shape[0]), replace=False
)
X_sample_email = X_test_email[sample_idx_email]

explainer_email = shap.TreeExplainer(email_model)
shap_values_email = explainer_email.shap_values(X_sample_email)

if isinstance(shap_values_email, list):
    shap_values_email = shap_values_email[1]

print("Generating summary plot (Email)...")
shap.summary_plot(
    shap_values_email, X_sample_email,
    feature_names=combined_feature_names,  # from Piece 3b
    show=False, max_display=20
)
plt.title("SHAP Summary — Email Features (Handcrafted + TF-IDF, XGBoost)")
plt.tight_layout()
plt.savefig(OUT_DIR + 'shap_summary_email.png', dpi=150, bbox_inches='tight')
plt.show()

# -----------------------------------------------------------
# 7d. Extract top-N most important features per dataset as a
#     clean table for your report (not just a plot)
# -----------------------------------------------------------
def top_shap_features(shap_values, feature_names, n=15):
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    return pd.Series(mean_abs_shap, index=feature_names).sort_values(ascending=False).head(n)

print("\n=== Top 15 SHAP features — URL_Lexical ===")
top_url_shap = top_shap_features(shap_values_url, X_sample_url.columns, n=15)
print(top_url_shap)

print("\n=== Top 15 SHAP features — Email ===")
top_email_shap = top_shap_features(shap_values_email, combined_feature_names, n=15)
print(top_email_shap)

top_url_shap.to_csv(OUT_DIR + 'shap_top_features_url.csv')
top_email_shap.to_csv(OUT_DIR + 'shap_top_features_email.csv')

print("\nSaved SHAP plots and top-feature tables to Drive.")

In [ ]:
# =========================================================
# QUICK CHECK: Remove likely corpus-artifact tokens, refit TF-IDF
# =========================================================
CORPUS_ARTIFACT_WORDS = ['enron', 'wrote', 'university', 'linguistics', 'thanks']

tfidf_clean = TfidfVectorizer(
    max_features=500, ngram_range=(1, 2), stop_words='english',
    min_df=5, max_df=0.9
)
# Add artifact words to stopword list
tfidf_clean.stop_words_ = None  # reset internal cache
custom_stopwords = list(tfidf_clean.get_params()['stop_words'] or []) if isinstance(tfidf_clean.get_params()['stop_words'], list) else []
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
extended_stopwords = list(ENGLISH_STOP_WORDS) + CORPUS_ARTIFACT_WORDS

tfidf_clean = TfidfVectorizer(
    max_features=500, ngram_range=(1, 2), stop_words=extended_stopwords,
    min_df=5, max_df=0.9
)
X_tfidf_clean = tfidf_clean.fit_transform(df_email_feat['text_clean'])

X_email_combined_clean = hstack([X_email_handcrafted_sparse, X_tfidf_clean]).tocsr()

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_email_combined_clean, y_email, test_size=0.2, stratify=y_email, random_state=RANDOM_STATE
)

quick_check = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                             eval_metric='logloss', random_state=RANDOM_STATE)
quick_check.fit(X_train_c, y_train_c)
acc_clean = accuracy_score(y_test_c, quick_check.predict(X_test_c))
print(f"Accuracy without corpus-artifact words: {acc_clean:.4f}")
print(f"(Original XGBoost accuracy was: 0.9661)")

In [ ]:
# =========================================================
# LOCK IN: Retrain all 4 base models on clean email features
# =========================================================

# Update the "official" email feature set going forward
X_email_combined = X_email_combined_clean
combined_feature_names = list(X_email.columns) + [f"tfidf_{w}" for w in tfidf_clean.get_feature_names_out()]

# Update splits dict so all later pieces use the clean version
splits['Email'] = (X_train_c, X_test_c, y_train_c, y_test_c)

# Retrain all 4 baseline models on the clean data
print("Retraining all 4 base models on clean Email features...")
email_results_clean = train_all_models(X_train_c, y_train_c, X_test_c, y_test_c, "Email (clean)")

# Overwrite the old (artifact-contaminated) results
all_results['Email'] = email_results_clean

# Save the clean vectorizer + updated feature matrix
with open(OUT_DIR + 'tfidf_vectorizer_clean.pkl', 'wb') as f:
    pickle.dump(tfidf_clean, f)

from scipy.sparse import save_npz
save_npz(OUT_DIR + 'email_features_combined_clean.npz', X_email_combined_clean)

with open(MODEL_DIR + 'combined_feature_names_clean.pkl', 'wb') as f:
    pickle.dump(combined_feature_names, f)

print("\n=== Updated baseline accuracy (Email, clean features) ===")
for name, res in email_results_clean.items():
    acc = (res['y_pred'] == y_test_c).mean()
    print(f"{name}: {acc:.4f}")

In [ ]:
# =========================================================
# Rerun stacking for Email only (now using clean features)
# =========================================================
X_train, X_test, y_train, y_test = splits['Email']
stacking_results['Email'] = train_stacking_ensemble(
    'Email', X_train, y_train, X_test, y_test
)

# Update saved model
with open(f"{MODEL_DIR}Email_StackingEnsemble.pkl", 'wb') as f:
    pickle.dump(stacking_results['Email']['model'], f)

print("Email stacking ensemble updated with clean features.")

In [ ]:
# =========================================================
# PIECE 8: Final Evaluation — Confusion Matrix, ROC, P/R/F1
# =========================================================
from sklearn.metrics import (confusion_matrix, classification_report,
                              roc_curve, auc, precision_recall_fscore_support)

# -----------------------------------------------------------
# 8a. Collect final model set per feature set — best individual
#     + stacking ensemble, for direct comparison
# -----------------------------------------------------------
final_models = {}
for feat_name in ['URL_Lexical', 'URL_Full', 'Email']:
    _, X_test, _, y_test = splits[feat_name]
    best_name = max(all_results[feat_name],
                     key=lambda m: accuracy_score(y_test, all_results[feat_name][m]['y_pred']))
    final_models[feat_name] = {
        'best_individual': (best_name, all_results[feat_name][best_name]),
        'stacking': ('StackingEnsemble', stacking_results[feat_name]),
    }

# -----------------------------------------------------------
# 8b. Full metrics table — accuracy, precision, recall, F1
#     for every model type, every feature set (your main
#     "accuracy comparison table" deliverable)
# -----------------------------------------------------------
metrics_rows = []
for feat_name in ['URL_Lexical', 'URL_Full', 'Email']:
    _, X_test, _, y_test = splits[feat_name]
    for model_name, res in all_results[feat_name].items():
        p, r, f1, _ = precision_recall_fscore_support(y_test, res['y_pred'], average='binary')
        metrics_rows.append({
            'Feature Set': feat_name, 'Model': model_name,
            'Accuracy': accuracy_score(y_test, res['y_pred']),
            'Precision': p, 'Recall': r, 'F1': f1
        })
    # Add stacking row
    stack_res = stacking_results[feat_name]
    p, r, f1, _ = precision_recall_fscore_support(y_test, stack_res['y_pred'], average='binary')
    metrics_rows.append({
        'Feature Set': feat_name, 'Model': 'StackingEnsemble',
        'Accuracy': stack_res['accuracy'], 'Precision': p, 'Recall': r, 'F1': f1
    })

final_metrics_table = pd.DataFrame(metrics_rows).sort_values(['Feature Set', 'F1'], ascending=[True, False])
print("=== Final Metrics Table (Accuracy, Precision, Recall, F1) ===")
print(final_metrics_table.to_string(index=False))
final_metrics_table.to_csv(OUT_DIR + 'final_metrics_table.csv', index=False)

# -----------------------------------------------------------
# 8c. Confusion matrices — best individual + stacking, per
#     feature set (6 total plots)
# -----------------------------------------------------------
fig, axes = plt.subplots(3, 2, figsize=(11, 15))

for row_idx, feat_name in enumerate(['URL_Lexical', 'URL_Full', 'Email']):
    _, X_test, _, y_test = splits[feat_name]

    for col_idx, (label, (name, res)) in enumerate(final_models[feat_name].items()):
        cm = confusion_matrix(y_test, res['y_pred'])
        ax = axes[row_idx, col_idx]
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                    xticklabels=['Legitimate', 'Phishing'],
                    yticklabels=['Legitimate', 'Phishing'], cbar=False)
        ax.set_title(f"{feat_name} — {name}")
        ax.set_xlabel('Predicted')
        ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig(OUT_DIR + 'confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

# -----------------------------------------------------------
# 8d. ROC curves — all models overlaid, per feature set
#     (3 plots, one per feature set, all models on each)
# -----------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, feat_name in enumerate(['URL_Lexical', 'URL_Full', 'Email']):
    _, X_test, _, y_test = splits[feat_name]
    ax = axes[idx]

    for model_name, res in all_results[feat_name].items():
        fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, label=f"{model_name} (AUC={roc_auc:.4f})")

    # Add stacking ensemble
    stack_res = stacking_results[feat_name]
    fpr, tpr, _ = roc_curve(y_test, stack_res['y_proba'])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f"Stacking (AUC={roc_auc:.4f})", linewidth=2.5, linestyle='--')

    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, linewidth=1)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'ROC Curve — {feat_name}')
    ax.legend(loc='lower right', fontsize=8)

plt.tight_layout()
plt.savefig(OUT_DIR + 'roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# -----------------------------------------------------------
# 8e. Full classification reports (text form) for your
#     appendix / supplementary material
# -----------------------------------------------------------
for feat_name in ['URL_Lexical', 'URL_Full', 'Email']:
    _, X_test, _, y_test = splits[feat_name]
    print(f"\n{'='*60}\n{feat_name} — Stacking Ensemble Classification Report\n{'='*60}")
    print(classification_report(y_test, stacking_results[feat_name]['y_pred'],
                                 target_names=['Legitimate', 'Phishing']))

print("\nAll evaluation artifacts saved to Drive:")
print("- final_metrics_table.csv")
print("- confusion_matrices.png")
print("- roc_curves.png")

In [ ]:
%%writefile feature_engineering.py
import re
import math
from collections import Counter
import numpy as np

SUSPICIOUS_KEYWORDS = ['login', 'secure', 'account', 'verify', 'update',
                        'confirm', 'bank', 'signin', 'security', 'ebay',
                        'paypal', 'webscr']

URGENCY_KEYWORDS = ['urgent', 'immediately', 'verify', 'suspend', 'click here',
                     'act now', 'limited time', 'confirm', 'password',
                     'security alert', 'account will be', 'update your',
                     'winner', 'congratulations', 'free', 'click below']

def url_entropy(url):
    if not url:
        return 0
    counts = Counter(url)
    length = len(url)
    return -sum((c / length) * math.log2(c / length) for c in counts.values())

def extract_url_features(url: str) -> dict:
    return {
        'URLLength': len(url),
        'NoOfLettersInURL': sum(c.isalpha() for c in url),
        'NoOfDegitsInURL': sum(c.isdigit() for c in url),
        'URLEntropy': url_entropy(url),
        'HasSuspiciousKeyword': int(any(kw in url.lower() for kw in SUSPICIOUS_KEYWORDS)),
        # add every other column exactly as computed in Piece 3
    }

def clean_for_tfidf(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' URLTOKEN ', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def extract_email_features(text: str) -> dict:
    words = text.split()
    return {
        'text_length': len(text),
        'word_count': len(words),
        'num_links': len(re.findall(r'https?://\S+|www\.\S+', text)),
        'num_exclamations': text.count('!'),
        'num_digits': sum(c.isdigit() for c in text),
        'urgency_keyword_count': sum(text.lower().count(kw) for kw in URGENCY_KEYWORDS),
        # add the rest exactly as computed in Piece 3
    }

In [ ]:
import shutil, os

DEPLOY_DIR = '/content/drive/MyDrive/phishing_project/deploy_artifacts/'
os.makedirs(DEPLOY_DIR, exist_ok=True)

# Copy final chosen models (pick ONE final model per feature set —
# e.g. best individual or stacking ensemble, whichever you're shipping)
shutil.copy(MODEL_DIR + 'URL_Lexical_StackingEnsemble.pkl', DEPLOY_DIR)
shutil.copy(MODEL_DIR + 'Email_StackingEnsemble.pkl', DEPLOY_DIR)
shutil.copy(OUT_DIR + 'tfidf_vectorizer_clean.pkl', DEPLOY_DIR)

# Save the exact list/order of feature columns each model expects —
# critical for the app to build feature vectors in the right order
import pickle
with open(DEPLOY_DIR + 'url_feature_columns.pkl', 'wb') as f:
    pickle.dump(list(X_url_lexical.columns), f)
with open(DEPLOY_DIR + 'email_feature_columns.pkl', 'wb') as f:
    pickle.dump(list(X_email.columns), f)  # handcrafted only, TF-IDF handled by vectorizer

print("Deployment artifacts ready in:", DEPLOY_DIR)
print(os.listdir(DEPLOY_DIR))

In [ ]:
import shutil, os, json, datetime

DEPLOY_DIR = '/content/drive/MyDrive/phishing_project/deploy_artifacts/'
os.makedirs(DEPLOY_DIR, exist_ok=True)

shutil.copy(MODEL_DIR + 'URL_Lexical_StackingEnsemble.pkl', DEPLOY_DIR)
shutil.copy(MODEL_DIR + 'Email_StackingEnsemble.pkl', DEPLOY_DIR)
shutil.copy(OUT_DIR + 'tfidf_vectorizer_clean.pkl', DEPLOY_DIR)

with open(DEPLOY_DIR + 'url_feature_columns.pkl', 'wb') as f:
    pickle.dump(list(X_url_lexical.columns), f)
with open(DEPLOY_DIR + 'email_feature_columns.pkl', 'wb') as f:
    pickle.dump(list(X_email.columns), f)

version_info = {'version': 'v1.0', 'trained_by': 'Shankha',
                 'date': str(datetime.date.today())}
with open(DEPLOY_DIR + 'version_info.json', 'w') as f:
    json.dump(version_info, f, indent=2)

print("Done. Files in deploy_artifacts/:", os.listdir(DEPLOY_DIR))